In [1]:
import cv2
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from IMMTracker import *

from ultralytics import YOLO
import albumentations as A
import torch
import torchvision
import torchaudio

In [2]:
def fx_ballistic(x, dt):
    """Балістична нелінійна модель: x_next = F*x * gravity_effect
    F-матриця для [x, vx, y, vy, z, vz]"""
    vx, vy, vz = x[1], x[3], x[5]
    vx = np.clip(vx, -40.0, 40.0)
    vy = np.clip(vy, -40.0, 40.0)
    vz = np.clip(vz, -40.0, 40.0)

    F = np.array([[1, dt, 0, 0,  0,  0],                  # x
                  [0, 1,  0, 0,  0,  0],                  # vx
                  [0, 0,  1, dt, 0,  0],                  # y
                  [0, 0,  0, 1,  0,  0],                  # vy
                  [0, 0,  0, 0,  1,  dt],                 # z
                  [0, 0,  0, 0,  0,  1]], dtype=float)    # vz

    B = np.diag([0.5 * dt**2, dt, 0.5 * dt**2, dt, 0.5 * dt**2, dt])
    k = 0.0314
    v_mag = np.sqrt(x[1]**2 + x[3]**2 + x[5]**2)
    a_dx = -k * v_mag * vx
    a_dy = -k * v_mag * vy
    a_dz = -k * v_mag * vz
    u = np.array([a_dx, a_dx, a_dy - 9.81, a_dy - 9.81, a_dz, a_dz])

    x_next = np.dot(F, x) + np.dot(B, u)

    return x_next

def fx_hit(x, dt):
    """
    Модель удару: Constant Velocity.
    Стан: [x, vx, y, vy, z, vz]
    """
    F = np.array([[1, dt, 0, 0,  0,  0],
                  [0, 1,  0, 0,  0,  0],
                  [0, 0,  1, dt, 0,  0],
                  [0, 0,  0, 1,  0,  0],
                  [0, 0,  0, 0,  1,  dt],
                  [0, 0,  0, 0,  0,  1]], dtype=float)

    return np.dot(F, x)

def fx_bounce(x, dt):
    """Модель відскоку, марковська матриця"""
    epsilon = 0.75 # Коефіцієнт реституції
    friction = 0.85 # Коефіцієнт тертя
    x_next = np.copy(x)
    vx_new = x[1] * friction
    vy_new = -x[3] * epsilon
    vz_new = x[5] * friction

    x_next[0] += vx_new * dt
    x_next[2] += vy_new * dt
    x_next[4] += vz_new * dt

    x_next[1] = vx_new
    x_next[3] = vy_new
    x_next[5] = vz_new

    return x_next

def hx(x):
    return np.array([x[0], x[2], x[4]])

def measurement_transform(prediction_data, K, R_matrix, camera_pos, ball_diameter=0.21):
    u_cam, v_cam, w_cam = prediction_data['x_pos'], prediction_data['y_pos'], prediction_data['w_box']
    raw_x, raw_y, raw_z = get_3d_position(u_cam, v_cam, w_cam, K, R_matrix, camera_pos, ball_diameter)
    raw_y = -raw_y
    measurement = np.array([raw_x, raw_y, raw_z], dtype=np.float32)

    return measurement

def get_dynamic_transition_matrix(y, vy):
    """
    Повертає марковську матрицю 3x3 для моделей: [Ballistic, Hit, Bounce]
    """
    M = np.array([[0.95, 0.04, 0.01],
                  [0.60, 0.40, 0.00],
                  [0.90, 0.00, 0.10]])

    if y < 0.3 and vy < 0:
        M[0] = [0.10, 0.05, 0.85]
        M[1] = [0.10, 0.05, 0.85]

    return M

In [3]:
pts_real_3d = np.array([
    [0.0, 0.0, 0.0],
    [9.0, 0.0, 0.0],
    [9.0, 18.0, 0.0],
    [0.0, 18.0, 0.0]],dtype=np.float32)

pts_video_2d = np.array([
    [73, 1031],
    [1835, 1027],
    [1504, 606],
    [405, 606]],dtype=np.float32)

K = np.array([
    [1300.0, 0.0,    960.0],
    [0.0,    1300.0, 540.0],
    [0.0,    0.0,    1.0]],dtype=np.float32)

R, tvec, camera_pos = calibrate_camera(pts_real_3d, pts_video_2d, K, dist=np.zeros((4, 1)))

# Спільні параметри
frames_per_second = 50
dt = 1 / frames_per_second
dim_x = 6
dim_z = 3
points = MerweScaledSigmaPoints(n=dim_x, alpha=.1, beta=2., kappa=1.)
P_init = np.diag([0.1, 50.0, 0.1, 50.0, 0.1, 50.0])
R_init = np.diag([0.01, 0.01, 0.04])

# Для балістичної моделі Q мінімальна
q_var_ballistic = 0.1
q_b = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_ballistic)
Q_ballistic = block_diag(q_b, q_b, q_b)

# Для моделі удару Q велика
q_var_hit = 100
q_h = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_hit)
Q_hit = block_diag(q_h, q_h, q_h)

# Для відскоку Q середня
q_var_bounce = 5
q_bnc = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_bounce)
Q_bounce = block_diag(q_bnc, q_bnc, q_bnc)

In [4]:
path_to_load = 'detect_infos/detection_data.json'
with open(path_to_load, 'r', encoding='utf-8') as f:
    raw_detections = json.load(f)

detections_df = pd.DataFrame.from_dict(raw_detections)
detections_df.set_index('frame', inplace=True)
detections_df["ball_detected"] = detections_df["ball_detected"].fillna(False)

In [5]:
# main_model = YOLO('/home/var-roman/Desktop/my_projects/diploma_project/training_models/models/main_model_april.pt')
#
# results = main_model.predict(
#     source='/home/var-roman/Desktop/my_projects/diploma_project/data/videos/Japan_vs_Poland_ultrashort.mp4',
#     imgsz=1920,
#     conf=0.4,
#     iou=0.45,
#     save=False,
#     stream=True
# )
#
# raw_detections = []
# frame_count = 0
#
# for r in results:
#     frame_count += 1
#
#     if len(r.boxes) > 0:
#         box = r.boxes[0].xywh[0]
#         raw_detections.append({
#             'ball_detected': True,
#             'frame': frame_count,
#             'x_pos': float(box[0]),
#             'y_pos': float(box[1]),
#             'w_box': float(box[2]),
#         })
#     else:
#         raw_detections.append({'ball_detected': False, 'frame': frame_count})
#
# path_to_save = 'detect_infos/raw_detection_data.json'
# with open(path_to_save, 'w', encoding='utf-8') as f:
#     json.dump(raw_detections, f, ensure_ascii=False, indent=4)

In [6]:
detections_df

,ball_detected,x_pos,y_pos,w_box
frame,,,,
1,False,NaN,NaN,NaN
2,False,NaN,NaN,NaN
3,False,NaN,NaN,NaN
4,False,NaN,NaN,NaN
5,False,NaN,NaN,NaN
...,...,...,...,...
1131,False,NaN,NaN,NaN
1132,False,NaN,NaN,NaN
1133,False,NaN,NaN,NaN


In [7]:
# Старий код для перевірки роботи однієї моделі UKF(балістичної) у вигляді тесту та виправлення помилок
# UKF = UnscentedKalmanFilter(dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_ballistic, hx=hx, points=points)
# UKF.P = np.diag([0.1, 50.0, 0.1, 50.0, 0.1, 50.0])
# UKF.Q = Q_ballistic
# UKF.R = np.diag([0.01, 0.01, 0.04])
#
# is_initialized = False
# smoothed_x, smoothed_y, smoothed_z = [], [], []
#
# for i in detections_df.iterrows():
#     if not is_initialized:
#         if i[1]['ball_detected']:
#             z = measurement_transform(i[1], K, R, camera_pos)
#             UKF.x = np.zeros([6])
#             UKF.x[0], UKF.x[2], UKF.x[4] = z
#             is_initialized = True
#         else:
#             continue
#
#     UKF.predict()
#
#     if i[1]['ball_detected']:
#         z = measurement_transform(i[1], K, R, camera_pos)
#         UKF.update(z)
#     else:
#         UKF.update(None)
#
#     smoothed_x.append(UKF.x[0]), smoothed_y.append(UKF.x[2]), smoothed_z.append(UKF.x[4])

In [8]:
# # Ballistic filter
# ukf_ballistic = UnscentedKalmanFilter(name='ballistic UKF', dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_ballistic, hx=hx, points=points)
# ukf_ballistic.P = P_init
# ukf_ballistic.Q = Q_ballistic
# ukf_ballistic.R = R_init
#
# # Hit filter
# ukf_hit = UnscentedKalmanFilter(name='hit UKF', dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_hit, hx=hx, points=points)
# ukf_hit.P = P_init
# ukf_hit.Q = Q_hit
# ukf_hit.R = R_init
#
# # Bounce filter
# ukf_bounce = UnscentedKalmanFilter(name='bounce UKF', dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_bounce, hx=hx, points=points)
# ukf_bounce.P = P_init
# ukf_bounce.Q = Q_bounce
# ukf_bounce.R = R_init
#
# filters_lt = [ukf_ballistic, ukf_hit, ukf_bounce]
# mu = np.array([0.95, 0.04, 0.01])
# M_base = np.array([[0.95, 0.04, 0.01],
#                    [0.60, 0.40, 0.00],
#                    [0.90, 0.00, 0.10]])
#
# imm_base = IMMEstimator(filters_lt, mu, M_base)
#
# is_initialized = False
# smoothed_x, smoothed_y, smoothed_z = [], [], []
# h_matrix = np.array([
#     [1, 0, 0, 0, 0, 0],
#     [0, 0, 1, 0, 0, 0],
#     [0, 0, 0, 0, 1, 0]
# ], dtype=float)
#
# for i, row in detections_df.iterrows():
#     if not is_initialized:
#         if row['ball_detected']:
#             z = measurement_transform(row, K, R, camera_pos)
#             imm_base.x[0], imm_base.x[2], imm_base.x[4] = z
#             imm_base.filter_setx()
#             is_initialized = True
#         else:
#             smoothed_x.append(None), smoothed_y.append(None), smoothed_z.append(None)
#             continue
#
#     imm_base.predict()
#
#     if row['ball_detected']:
#         z = measurement_transform(row, K, R, camera_pos)
#
#         z_mean = np.dot(h_matrix, imm_base.x_prior)
#         S = np.dot(h_matrix, np.dot(imm_base.P_prior, h_matrix.T)) + np.diag([0.01, 0.01, 0.04]) # R матриця
#         y = z - z_mean
#
#         try:
#             S_inv = np.linalg.inv(S)
#             mahal_dist_sq = np.dot(y.T, np.dot(S_inv, y))
#         except np.linalg.LinAlgError:
#             mahal_dist_sq = 1e5
#
#         if mahal_dist_sq > 11.34:
#             imm_base.update(None)
#         else:
#             imm_base.update(z)
#     else:
#         imm_base.update(None)
#
#     smoothed_x.append(imm_base.x[0])
#     smoothed_y.append(imm_base.x[2])
#     smoothed_z.append(imm_base.x[4])
#
# detections_df[['smoothed_x', 'smoothed_y', 'smoothed_z']] = np.nan, np.nan, np.nan
# smoothed_values = np.column_stack([smoothed_x, smoothed_y, smoothed_z])
# detections_df[['smoothed_x', 'smoothed_y', 'smoothed_z']] = smoothed_values

In [38]:
tracker = IMMTracker(dt=frames_per_second, max_age=50, min_hits=3)
final_trajectory = []
is_initialized = False

for frame_idx, row in detections_df.iterrows():
    current_detections = []
    if not is_initialized:
        if row['ball_detected']:
            z = measurement_transform(row, K, R, camera_pos)
            if z[1] >= -0.5:
                current_detections.append(z)
        else:
            final_trajectory.append({
            'frame': frame_idx,
            'smoothed_x': None,
            'smoothed_y': None,
            'smoothed_z': None,
            'track_id': None
        })
            continue

    tracker.update(current_detections)
    # Додай це у свій цикл тестування
    print(f"Кадр {frame_idx}: Детекцій знайдено: {len(current_detections)}")
    for t in tracker.tracks:
        print(f"  Track ID {t.track_id} [{t.state}]: hits={t.hits}, time_since_update={t.time_since_update}")

    confirmed = tracker.get_confirmed_tracks()

    if len(confirmed) > 0:
        best_track = confirmed[0]
        final_trajectory.append({
            'frame': frame_idx,
            'smoothed_x': best_track.imm.x[0],
            'smoothed_y': best_track.imm.x[2],
            'smoothed_z': best_track.imm.x[4],
            'track_id': best_track.track_id
        })
    else:
        # Немає підтверджених треків
        final_trajectory.append({'frame': frame_idx, 'smoothed_x': None, 'smoothed_y': None, 'smoothed_z': None, 'track_id': None})

Кадр 22: Детекцій знайдено: 1
  Track ID 1 [Tentative]: hits=1, time_since_update=0
Кадр 23: Детекцій знайдено: 1
  Track ID 1 [Tentative]: hits=2, time_since_update=0
Кадр 24: Детекцій знайдено: 1
  Track ID 1 [Confirmed]: hits=3, time_since_update=0
Кадр 25: Детекцій знайдено: 1
  Track ID 1 [Confirmed]: hits=4, time_since_update=0
Кадр 26: Детекцій знайдено: 1
  Track ID 1 [Confirmed]: hits=5, time_since_update=0
Кадр 28: Детекцій знайдено: 1
  Track ID 1 [Confirmed]: hits=5, time_since_update=1
  Track ID 2 [Tentative]: hits=1, time_since_update=0
Кадр 30: Детекцій знайдено: 1
  Track ID 1 [Confirmed]: hits=6, time_since_update=0
  Track ID 2 [Tentative]: hits=1, time_since_update=1
Кадр 31: Детекцій знайдено: 1
  Track ID 1 [Confirmed]: hits=6, time_since_update=1
  Track ID 2 [Tentative]: hits=2, time_since_update=0
Кадр 32: Детекцій знайдено: 1
  Track ID 1 [Confirmed]: hits=6, time_since_update=2
  Track ID 2 [Confirmed]: hits=3, time_since_update=0
Кадр 86: Детекцій знайдено: 

In [21]:
full_detections_df = detections_df.join(pd.DataFrame(final_trajectory).set_index('frame'), on='frame', how='left')

In [22]:
full_detections_df

,ball_detected,x_pos,y_pos,w_box,smoothed_x,smoothed_y,smoothed_z,track_id
frame,,,,,,,,
1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1131,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1132,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1133,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
smoothed_x, smoothed_y, smoothed_z = full_detections_df[['smoothed_x', 'smoothed_y', 'smoothed_z']].values.reshape((3, len(full_detections_df)))
fig = px.line_3d(x=smoothed_x, y=smoothed_z, z=smoothed_y, width=500, height=500)
fig.show()

In [35]:
type(list(full_detections_df[['smoothed_x', 'smoothed_y', 'smoothed_z']].values))

list